# Exercise Sheet 08

This markdown block is used to define macros for later markdown blocks. 
$\newcommand{\normaldist}{\mathcal{N}}$
$\newcommand{\betadist}{\text{Beta}}$
$\newcommand{\reals}{\mathbb{R}}$
$\newcommand{\ML}{\text{ML}}$
$\renewcommand{\vec}[1]{\boldsymbol{\mathbf{#1}}}$
$\newcommand{\matrix}[1]{\boldsymbol{\mathbf{#1}}}$
$\newcommand{\dataset}{\mathcal{D}}$
$\newcommand{\class}{\mathcal{C}}$
$\newcommand{\optimal}[1]{#1^{\star}}$
$\newcommand{\argmin}{\text{argmin}}$
$\newcommand{\argmax}{\text{argmax}}$
$\newcommand{\expct}{\mathbb{E}}$
$\newcommand{\entropy}[1]{H[#1]}$
$\newcommand{\conditionalentropy}[2]{\entropy{#1 | #2}}$
$\newcommand{\kldiv}[2]{KL(#1 || #2)}$
$\newcommand{\mutualinfo}[2]{I[#1;#2]}$
$\newcommand{\deriv}[2]{\frac{d}{d #2} \left( #1\right)}$
$\newcommand{\inputs}{\matrix{X}}$
$\newcommand{\identitymtx}{\matrix{I}}$
$\newcommand{\designmtx}{\matrix{\Phi}}$
$\newcommand{\featurevec}{\boldsymbol{\phi}}$
$\newcommand{\weights}{\vec{w}}$
$\newcommand{\inputvec}{\vec{x}}$
$\newcommand{\norm}[1]{\lVert #1 \rVert}$
$\newcommand{\grad}{\nabla}$

In [1]:
import math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import scipy.stats
import pandas as pd
from itertools import groupby

## ✏️ Exercise 8.1

Look at the slide **A Generative Approach**. Convince yourself that the posterior class probability $p(\class_{1}|\vec{x})$ shown on  can be represented as a logistic sigmoid:
$$\sigma(a(\mathbf{x})) = \frac{1}{1 + \exp(a(\mathbf{x}))}$$

where
$$a(\mathbf{x}) = \ln\left(\frac{p(\class_{1}|\vec{x})}{p(\class_{0}|\vec{x})}\right)$$

**Hint:** Begin by replacing $a(\mathbf{x})$ in the first equation with its form in the second equation. Then consider that $p(\class_{0}|\vec{x})$ can be written in terms of $p(\class_{1}|\vec{x})$, and $p(\class_{0})$ can be written in terms of $p(\class_{1})$.



## Exercise 8.2


This question explores the shared covariance generative model from the lectures.

**Before you start this question** make sure you have a copy of the data-file `iris.data` in the local directory or that you have changed the file location in the code block to point to the file in your system.

### 💻 8.2 a)

In module `fomlads.model.classification`, there is a function `shared_covariance_model_fit` that, when completed, should find the maximum-likelihood class priors, means, and shared covariance matrix for the generative model described in the lecture, see Results (8.2) and (8.3). Complete this function so it does just that?

*You may find it easier to partition the input matrix into class specific matrices, then find the maximum likelihood means and variances of each separately. For this, you can use the provided function `max_lik_mv_gaussian_approx` from `fomlads.model.density_estimation`.*

### 💻 8.2 b)

In the same module is a function `shared_covariance_model_predict` that, when completed, should predict the class for a matrix of input points. The output should be a vector of minimum-misclassification predictions
    $$\bar{\vec{y}} = (\bar{y}(\inputvec_{1}), \ldots, \bar{y}(\inputvec_{N}))$$
  where, for each row of your input $\inputvec_{n}^T$, you should predict $\bar{y}(\inputvec_{n}) = 1$ if $p(\class_{1} | \inputvec_{n},\pi, \vec{\mu}_{0}, \vec{\mu}_{1}, \matrix{S}) > 0.5$ and $\bar{y}(\inputvec_{n}) = 0$ otherwise. Complete this function so it does just that?

*Look at the slide **Shared Covariance $2$-Class Model** from the lecture. This gives you the joint probability, $p(x_n, \class_k)$ for our shared covariance model. You will need to apply Bayes Theorem to get the posterior class probabilities. To evaluate the probability density for a multivariate normal distribution, you can use the `multivariate_normal.pdf` function in the `scipy.stats` module.*



In [2]:
### Code to import classes for Ex. 8.2
from fomlads.data.external import import_for_classification
from fomlads.plot.exploratory import plot_scatter_array_classes
from fomlads.plot.exploratory import plot_class_histograms
from fomlads.plot.evaluations import plot_roc

from fomlads.model.classification import add_bias_column
from fomlads.model.classification import shared_covariance_model_fit
from fomlads.model.classification import shared_covariance_model_predict
from fomlads.model.classification import logistic_regression_fit
from fomlads.model.classification import logistic_regression_predict
from fomlads.model.classification import logistic_regression_prediction_probs
ifname = 'iris.data'
classes=['Iris-virginica','Iris-versicolor']
inputs, targets, field_names, classes = import_for_classification(
    ifname, classes=classes)
plot_scatter_array_classes(
    inputs, targets, field_names=field_names, classes=classes)

FileNotFoundError: [Errno 2] No such file or directory: 'iris.data'

In [ ]:
### Ex. 8.2 provided code
### Run once library updated
def fit_and_plot_roc_generative(inputs, targets, fig_ax=None, colour=None):
    """
    Takes input and target data for classification and fits shared covariance
    model. Then plots the ROC corresponding to the fit model.

    parameters
    ----------
    inputs - a 2d input matrix (array-like), each row is a data-point
    targets - 1d target vector (array-like) -- can be at most 2 classes ids
        0 and 1
    """
    pi, mean0, mean1, covmtx = shared_covariance_model_fit(inputs, targets)
    thresholds = np.linspace(0,1,101)
    N = targets.size
    num_neg = np.sum(1-targets)
    num_pos = np.sum(targets)
    false_positive_rates = np.empty(thresholds.size)
    true_positive_rates = np.empty(thresholds.size)
    for i, threshold in enumerate(thresholds):
        predicts = shared_covariance_model_predict(
            inputs, threshold, mean0, mean1, covmtx)
        num_false_positives = np.sum((predicts == 1) & (targets == 0))
        num_true_positives = np.sum((predicts == 1) & (targets == 1))
        false_positive_rates[i] = np.sum(num_false_positives)/num_neg
        true_positive_rates[i] = np.sum(num_true_positives)/num_pos
    fig, ax = plot_roc(
        false_positive_rates, true_positive_rates, fig_ax=fig_ax, colour=colour)
    # and for the class prior we learnt from the model
    predicts = shared_covariance_model_predict(
          inputs, pi, mean0, mean1, covmtx)
    fpr = np.sum((predicts == 1) & (targets == 0))/num_neg
    tpr = np.sum((predicts == 1) & (targets == 1))/num_pos
    ax.plot([fpr], [tpr], 'rx', markersize=8, markeredgewidth=2)
    return fig, ax


fit_and_plot_roc_generative(inputs, targets, colour='g')

## 💡 Exercise 8.3

*This exercise doesn't require you to write anything, just to think loosely about the problem.*

Look at the image (a) below. Would you recommend using the shared covariance model, the independent gaussian model and or logistic regression to train a classifier on this data. Would you use a feature mapping to help? If so, what how might you construct a good feature mapping? 

![difficult classes](non_linearly_separated_classes.png)

The other two datasets are unlikely to be tractable with any of these approaches (not without a good deal of engineering of feature mapping). Have you encountered other classification approaches that might perform better on either of these problems?

## Exercise 8.4

### 💡 8.4 a)

Look at the image below and imagine that this is a function $f(\vec{z})$ that you are going to optimise with a randomly chosen initial estimate of the maximum $\vec{z}^{(0)}$. How would you expect simple gradient ascent to perform on this data? What about the Newton-Raphson method? Explain your answer.
![odd contours 1](tutorial08_odd_contours1.png)


### 💡 8.4 b)

Consider the image below and imagine that this is another such function $f(\vec{z})$ that you aim to optimise under similar conditions. How would each method perform here? Explain your answer.
![odd contours 2](tutorial08_odd_contours2.png)



## Exercise 8.5

### 💻 8.5 a)

Now think about logistic regression applied to the same two classes: `Iris-virginica` and `Iris-versicolor` as in **Ex. 8.2**. Do you think this will perform any better or worse? Why? 

There are another two functions `logistic_regression_fit` and `logistic_regression_predict` in `fomlads.model.classification` which respectively fit and predict with a logistic regression model. Initially, `logistic_regression_fit` uses simple gradient ascent to learn the weights. Improve the algorithm using the Hessian matrix as described in the slides.

Now, fit some weights to the classes and evaluate your solution. 

### 💻 8.5 b)

Try it with the linearly separable classes too. What happens? Can you explain why?

### 💡 8.5 c)

On what sorts of data would you expect logistic regression to out-perform the shared covariance model and vice versa? Are there any methods from the regression lectures you might be able to adapt logistic regression, so that it avoids singular matrix errors?


In [ ]:
### Ex. 8.5 


## 💡Exercise 8.6

Consider one of the other datasets you have looked at during the sessions.

Imagine you were going to construct a classifier to distingiush between two of the classes.
* What classifier would you use?
* Would you define any sort of feature mapping?
* What parameters would you need to specify for this method?
* How would you evaluate performance?

**I am not asking you to try doing this. Simply think about the task.**
